# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop')]
print(f"Count: {len(pop_rows)}")
print(pop_rows)

Count: 3
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [ ]:
# Based on the sample data, PT-12, SKU#459812, and POPTART-STRAW all refer to the strawberry Pop-Tart variant
# Non-strawberry Pop-Tart rows must stay separate because they represent unique stock-keeping units with individual inventory and sales tracking

**Pop-Tart rows that are a different product:** _..._ any poptarts that are not strawberry

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [3]:
canonical_mapping = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW'
}

cat['canonical_sku'] = cat['sku'].map(canonical_mapping).fillna(cat['sku'])

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [4]:
print("Before normalization:")
print(cat['category'].value_counts())

cat['category'] = cat['category'].str.strip().str.title()

print("\nAfter normalization:")
print(cat['category'].value_counts())

Before normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

After normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [5]:
import re

def normalize_name(name):
    name = str(name).lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)


fragmented_groups = cat.groupby('normalized_name').filter(lambda x: x['sku'].nunique() > 1)
print(fragmented_groups[['sku', 'product_name', 'category']])

             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [6]:
assert cat['canonical_sku'].isnull().sum() == 0, "Error: Null values found in canonical_sku!"
assert len(cat) == len(cat['canonical_sku']), "Error: Row count changed!"
print("Assertion passed: Mapping is safe and complete.")

Assertion passed: Mapping is safe and complete.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a.) A lot of hygiene products like toothpaste and deoderant can often appear fragmented in similar ways to pop-tarts. You can fragment them based on brand, scent/type, or size.

b.) Consolidation becomes a business decision when items appear nearly identical in text description but carry differences in price, pack size, target retail channel, or store-brand vs. name-brand distinction. Merging a store brand products with a brand name ones based solely on matching  text would affect sales tracking. You would have to consult individuals that directly work with the products to verify the differences.
> Add blockquote

